# Vintage CORE Evaluation

Run one or all four base models on full **Original CORE**, **Filtered CORE**, and **Restyled CORE** from the `Vintage-CORE` branch of `zachnorton14/think.nano`. Models run sequentially in isolated subprocesses and write to separate result directories. Original CORE has 22 tasks; the Vintage bundles have 20, so the notebook reports both each bundle's native aggregate and a directly comparable Common-20 aggregate. LAMBADA is informational and never triggers fallback or blocks a result.

Results persist in Google Drive by default. The clean-1930s run also evaluates the complete 20,971,520-token validation split and appends full validation BPB plus every CORE result to its existing W&B run.

## 1. Configuration

The defaults run only the new clean-1930s model and chart it alongside any previously completed models found in the persistent results directory. Set `RUN_ALL_MODELS=True` only if you need to rerun missing models. GPT-1900 d34 and the full clean-1930s validation pass should use an A100-class Colab runtime.

In [ ]:
CLEAN_MODEL_ID = "clean1930s-d24-r12-ctx4096-sssl-fulltok-v1"
MODEL_ID = CLEAN_MODEL_ID
RUN_ALL_MODELS = False
RUN_FULL = True
RUN_FULL_VAL_BPB = True
FULL_VAL_BATCH_SIZE = 8
LOG_TO_WANDB = True
SAVE_TO_DRIVE = True
DOWNLOAD_RESULTS = False

VALID_MODELS = [
    "think-d12-r30",
    "modern-d24",
    "gpt1900-d34",
    CLEAN_MODEL_ID,
]
assert MODEL_ID in VALID_MODELS
assert RUN_FULL is True, "The publication runs are full evaluations."
MODELS_TO_RUN = VALID_MODELS if RUN_ALL_MODELS else [MODEL_ID]
MODELS_TO_CHART = VALID_MODELS
print("Models to run:", ", ".join(MODELS_TO_RUN))
print("Charts will include every completed model found among:", ", ".join(MODELS_TO_CHART))

## 2. Environment

Select **Runtime → Change runtime type → GPU** first. Add `HF_TOKEN` and `WANDB_API_KEY` in Colab Secrets (key icon at left). The tokens are read without displaying their values.

In [ ]:
import json, os, platform, queue, shutil, subprocess, sys, threading, time
from pathlib import Path
import torch

assert torch.cuda.is_available(), "A CUDA GPU is required."
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}")
print(f"VRAM: {props.total_memory / 2**30:.1f} GiB")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
if ({"gpt1900-d34", CLEAN_MODEL_ID} & set(MODELS_TO_RUN)) and "A100" not in props.name:
    print("WARNING: GPT-1900 d34 and full clean-1930s validation are intended for an A100-class runtime.")

REPO_URL = "https://github.com/zachnorton14/think.nano.git"
REPO_BRANCH = "Vintage-CORE"
repo = "/content/think.nano"
if not os.path.exists(repo):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, repo], check=True)
else:
    subprocess.run(["git", "-C", repo, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", repo, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", repo, "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
branch = subprocess.check_output(["git", "-C", repo, "branch", "--show-current"], text=True).strip()
assert branch == REPO_BRANCH, f"Expected {REPO_BRANCH}, found {branch}"
print(f"Repository: {REPO_URL} @ {branch}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{repo}/dev/vintage_core_colab/requirements.txt", "pyarrow"], check=True)

try:
    from google.colab import userdata
    for secret_name in ("HF_TOKEN", "WANDB_API_KEY"):
        secret_value = userdata.get(secret_name)
        if secret_value:
            os.environ[secret_name] = secret_value
except Exception:
    pass
print("HF_TOKEN available:", bool(os.environ.get("HF_TOKEN")))
print("WANDB_API_KEY available:", bool(os.environ.get("WANDB_API_KEY")))

# Build a disposable evaluator directory so the checked-out repository stays untouched.
source_eval_dir = Path(repo) / "dev/vintage_core_colab"
notebook_eval_dir = Path("/content/vintage-core-notebook-runtime")
notebook_eval_dir.mkdir(parents=True, exist_ok=True)
for filename in ("vintage_core_eval.py", "models.json", "bundles.json"):
    shutil.copy2(source_eval_dir / filename, notebook_eval_dir / filename)
EVALUATOR_PATH = notebook_eval_dir / "vintage_core_eval.py"
registry_path = notebook_eval_dir / "models.json"
registry = json.loads(registry_path.read_text())
clean_prefix = "experiments/clean1930s-d24-r12-ctx4096-sssl-fulltok-v1"
registry["models"][CLEAN_MODEL_ID] = {
    "display_name": "clean 1930s d24 r12 ctx4096 SSSL full-token",
    "artifact_repo": "jbduran/think.nano",
    "artifact_revision": "cefb9156ea1d856686425566b60d0c6fe7b8f1de",
    "checkpoint": f"{clean_prefix}/base_checkpoints/model_008352.pt",
    "metadata": f"{clean_prefix}/base_checkpoints/meta_008352.json",
    "tokenizer_dir": f"{clean_prefix}/tokenizer",
    "allow_patterns": [
        f"{clean_prefix}/base_checkpoints/model_008352.pt",
        f"{clean_prefix}/base_checkpoints/meta_008352.json",
        f"{clean_prefix}/tokenizer/tokenizer.pkl",
    ],
    "runtime": {
        "type": "git",
        "url": "https://github.com/zachnorton14/think.nano.git",
        "revision": "c850ee9513ff3af301af2b7ff127e8709a4f6d49",
    },
    "recommended_gpu": "A100 recommended",
}
registry_path.write_text(json.dumps(registry, indent=2) + "\n")
print("Registered clean model in the disposable notebook evaluator; repository checkout is unchanged.")

# GPT-1900 uses Michael Hla's bundled nanochat runtime, never think.nano's.
if "gpt1900-d34" in MODELS_TO_RUN:
    from huggingface_hub import snapshot_download
    michael_runtime = Path(snapshot_download(
        repo_id="mhla/gpt1900-d34-22btok",
        revision="d6330f9f0a17ce13da36fb951d7987bb03e6fbd0",
        allow_patterns=["nanochat/**"],
        cache_dir=os.path.expanduser("~/.cache/vintage-core/huggingface"),
        token=os.environ.get("HF_TOKEN"),
    ))
    required_runtime_files = ["gpt.py", "optim.py", "common.py", "flash_attention.py", "tokenizer.py"]
    missing_runtime_files = [name for name in required_runtime_files if not (michael_runtime / "nanochat" / name).is_file()]
    assert not missing_runtime_files, f"Michael Hla runtime is incomplete: {missing_runtime_files}"
    print("Prepared Michael Hla GPT-1900 runtime:", michael_runtime)

## 3. Persistent results and CORE evaluation

The notebook mounts Google Drive by default and stores results under `MyDrive/vintage-core-results`. Before running, it salvages completed files from either older `/content/vintage-core-results` layout. Each bundle JSON is written immediately after that bundle finishes.

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = "/content/drive/MyDrive/vintage-core-results"
else:
    RESULTS_ROOT = "/content/vintage-core-results"
Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)

# Salvage results from both earlier notebook layouts if this Colab VM is still alive.
for legacy_name in ("/content/vintage-core-results", "/content/vintage-core-all-results"):
    legacy_root = Path(legacy_name)
    if not legacy_root.exists() or legacy_root.resolve() == Path(RESULTS_ROOT).resolve():
        continue
    if (legacy_root / "summary.csv").is_file():
        shutil.copytree(legacy_root, Path(RESULTS_ROOT) / "think-d12-r30", dirs_exist_ok=True)
        print(f"Salvaged legacy single-model results from {legacy_root}")
    for model_id in VALID_MODELS:
        source = legacy_root / model_id
        if source.is_dir():
            shutil.copytree(source, Path(RESULTS_ROOT) / model_id, dirs_exist_ok=True)
            print(f"Salvaged {model_id} from {legacy_root}")

def stream_command(command, cwd=None, extra_env=None):
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    if extra_env:
        environment.update(extra_env)
    process = subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=environment,
    )
    assert process.stdout is not None
    output_queue = queue.Queue()
    def pump_output():
        for line in process.stdout:
            output_queue.put(line)
        output_queue.put(None)
    threading.Thread(target=pump_output, daemon=True).start()
    started = time.monotonic()
    while True:
        try:
            line = output_queue.get(timeout=30)
        except queue.Empty:
            elapsed_minutes = (time.monotonic() - started) / 60
            print(f"[still running: {elapsed_minutes:.1f} minutes elapsed]", flush=True)
            continue
        if line is None:
            break
        print(line, end="", flush=True)
    return process.wait()

RUN_STATUS = {}
for model_id in MODELS_TO_RUN:
    output_dir = f"{RESULTS_ROOT}/{model_id}"
    if os.path.isfile(f"{output_dir}/summary.csv"):
        print(f"ALREADY COMPLETE: {model_id}")
        RUN_STATUS[model_id] = 0
        continue
    command = [
        sys.executable, "-u", str(EVALUATOR_PATH),
        "--model", model_id,
        "--bundles", "original,filtered,restyled",
        "--output-dir", output_dir,
        "--max-per-task", "-1",
    ]
    print("\nRunning:", " ".join(command), flush=True)
    returncode = stream_command(command)
    RUN_STATUS[model_id] = returncode
    if returncode:
        print(f"FAILED: {model_id} (exit {returncode}). Scroll up to the first traceback for the cause.")
    else:
        print(f"COMPLETED: {model_id}")
print("Run status:", RUN_STATUS)
print("Persistent results directory:", RESULTS_ROOT)

## 4. Full clean-1930s validation BPB

This is a separate full 20,971,520-token validation evaluation, not the 2,097,152-token periodic training estimate. It uses validation shard `shard_00472.parquet`, the checkpoint's exact runtime commit, and writes `full_val_bpb.json`. When W&B logging is enabled, the evaluator resumes W&B run `e69c1e59` and appends `eval/full_val_bpb` plus the per-position BPB curve.

In [ ]:
FULL_VAL_TOKENS = 20_971_520
CLEAN_RUNTIME_REVISION = "c850ee9513ff3af301af2b7ff127e8709a4f6d49"
CLEAN_ARTIFACT_REVISION = "cefb9156ea1d856686425566b60d0c6fe7b8f1de"
CLEAN_WANDB_RUN_ID = "e69c1e59"
CLEAN_WANDB_ENTITY = "jbduran-thinkingmachinesncsu"
CLEAN_WANDB_PROJECT = "think.nano"
clean_output_dir = Path(RESULTS_ROOT) / CLEAN_MODEL_ID
full_val_path = clean_output_dir / "full_val_bpb.json"

if RUN_FULL_VAL_BPB and CLEAN_MODEL_ID in MODELS_TO_RUN:
    if full_val_path.is_file():
        print("ALREADY COMPLETE: full clean-1930s validation BPB")
    else:
        assert os.environ.get("HF_TOKEN"), "HF_TOKEN is required to download the private artifacts/dataset."
        if LOG_TO_WANDB:
            assert os.environ.get("WANDB_API_KEY"), "WANDB_API_KEY is required when LOG_TO_WANDB=True."
        from huggingface_hub import snapshot_download
        cache_dir = os.path.expanduser("~/.cache/vintage-core/huggingface")
        clean_snapshot = Path(snapshot_download(
            repo_id="jbduran/think.nano",
            revision=CLEAN_ARTIFACT_REVISION,
            allow_patterns=[
                f"{clean_prefix}/base_checkpoints/model_008352.pt",
                f"{clean_prefix}/base_checkpoints/meta_008352.json",
                f"{clean_prefix}/tokenizer/tokenizer.pkl",
            ],
            cache_dir=cache_dir, token=os.environ.get("HF_TOKEN"),
        ))
        val_snapshot = Path(snapshot_download(
            repo_id="jbduran/think-dataset-clean-1930s", repo_type="dataset",
            revision="main", allow_patterns=["shard_00472.parquet"],
            cache_dir=cache_dir, token=os.environ.get("HF_TOKEN"),
        ))
        assert (val_snapshot / "shard_00472.parquet").is_file()

        clean_runtime = Path("/content/think.nano-clean-runtime-c850ee9")
        if not (clean_runtime / ".git").is_dir():
            subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(clean_runtime)], check=True)
        subprocess.run(["git", "-C", str(clean_runtime), "fetch", "origin", CLEAN_RUNTIME_REVISION, "--depth", "1"], check=True)
        subprocess.run(["git", "-C", str(clean_runtime), "checkout", "--detach", CLEAN_RUNTIME_REVISION], check=True)

        clean_output_dir.mkdir(parents=True, exist_ok=True)
        command = [
            sys.executable, "-u", "-m", "scripts.base_eval",
            "--eval", "bpb",
            "--checkpoint-dir", str(clean_snapshot / clean_prefix / "base_checkpoints"),
            "--tokenizer-dir", str(clean_snapshot / clean_prefix / "tokenizer"),
            "--step", "8352",
            "--data-dir", str(val_snapshot),
            "--split", "val",
            "--split-tokens", str(FULL_VAL_TOKENS),
            "--device-batch-size", str(FULL_VAL_BATCH_SIZE),
            "--per-position-bpb",
            "--output-json", str(full_val_path),
        ]
        if LOG_TO_WANDB:
            command.extend(["--wandb-run-id", CLEAN_WANDB_RUN_ID, "--wandb-run-name", CLEAN_MODEL_ID])
        validation_steps = FULL_VAL_TOKENS // (FULL_VAL_BATCH_SIZE * 4096)
        print(f"Full validation plan: {FULL_VAL_TOKENS:,} tokens in {validation_steps:,} batches.")
        print("\nRunning full validation BPB:", " ".join(command), flush=True)
        returncode = stream_command(
            command, cwd=str(clean_runtime),
            extra_env={
                "PYTHONPATH": str(clean_runtime),
                "WANDB_PROJECT": CLEAN_WANDB_PROJECT,
                "WANDB_ENTITY": CLEAN_WANDB_ENTITY,
            },
        )
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
        print("COMPLETED: full validation BPB ->", full_val_path)
elif RUN_FULL_VAL_BPB:
    print("Full validation BPB skipped because the clean model is not in MODELS_TO_RUN.")

## 5. Results and comparison charts

These cells search the persistent directory for all four model IDs, regardless of which model was selected in this session. The aggregate results for the first three models were recovered from the saved July screenshot and evaluator transcript. They are written to `recovered_historical_summary.csv` on Drive and used only when a real per-model `summary.csv` is absent. A future rerun automatically takes precedence. Recovered aggregate rows do not pretend that missing GPT-1900 per-task files survived.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Historical aggregates recovered from the saved result screenshot. Think-d12 and
# modern-d24 retain extra precision from their complete evaluator transcript; GPT-1900
# is recoverable only to the four decimal places displayed in the screenshot.
RECOVERED_HISTORICAL_SUMMARY = pd.DataFrame([
    ["think-d12-r30", "original", 0.089531045, 0.072491750, 1822.9],
    ["think-d12-r30", "filtered", 0.074223800, 0.074223800, 1074.7],
    ["think-d12-r30", "restyled", 0.074149200, 0.074149200, 1077.3],
    ["modern-d24", "original", 0.260598500, 0.259738650, 4120.9],
    ["modern-d24", "filtered", 0.258870450, 0.258870450, 2475.4],
    ["modern-d24", "restyled", 0.253259000, 0.253259000, 2509.9],
    ["gpt1900-d34", "original", 0.1331, 0.1213, 7337.2],
    ["gpt1900-d34", "filtered", 0.1270, 0.1270, 4428.8],
    ["gpt1900-d34", "restyled", 0.1389, 0.1389, 4498.6],
], columns=["model", "bundle", "native_core", "common_20_core", "runtime_seconds"])
recovered_summary_path = Path(RESULTS_ROOT) / "recovered_historical_summary.csv"
RECOVERED_HISTORICAL_SUMMARY.to_csv(recovered_summary_path, index=False)
print("Persistent recovered summary:", recovered_summary_path)

summaries = []
SUMMARY_SOURCES = {}
for model_id in MODELS_TO_CHART:
    summary_path = Path(RESULTS_ROOT) / model_id / "summary.csv"
    if summary_path.is_file():
        frame = pd.read_csv(summary_path)
        frame["model"] = model_id
        summaries.append(frame)
        SUMMARY_SOURCES[model_id] = "evaluated JSON/CSV"
    else:
        recovered = RECOVERED_HISTORICAL_SUMMARY[RECOVERED_HISTORICAL_SUMMARY["model"] == model_id]
        if not recovered.empty:
            summaries.append(recovered.copy())
            SUMMARY_SOURCES[model_id] = "recovered screenshot/transcript aggregate"
            print(f"Using recovered aggregate summary: {model_id}")
        else:
            print(f"No saved or recovered CORE result yet: {model_id}")
if not summaries:
    raise FileNotFoundError(f"No completed summary.csv files under {RESULTS_ROOT}. Inspect the evaluator traceback above.")
summary_all = pd.concat(summaries, ignore_index=True)
summary_all = summary_all[["model", "bundle", "native_core", "common_20_core", "runtime_seconds"]]
summary_all.to_csv(f"{RESULTS_ROOT}/combined_summary.csv", index=False)
print("Summary provenance:", SUMMARY_SOURCES)
display(summary_all.style.format({"native_core": "{:.4f}", "common_20_core": "{:.4f}", "runtime_seconds": "{:.1f}"}))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for axis, metric, title in zip(axes, ["native_core", "common_20_core"], ["Native CORE", "Common-20 CORE"]):
    summary_all.pivot(index="model", columns="bundle", values=metric).plot(kind="bar", ax=axis)
    axis.set_title(title)
    axis.set_xlabel("")
    axis.set_ylabel("Centered CORE")
    axis.tick_params(axis="x", rotation=20)
    axis.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
accuracy_frames, delta_frames = [], []
DETAILED_MODELS = []
for model_id in MODELS_TO_CHART:
    model_dir = Path(RESULTS_ROOT) / model_id
    accuracy_path = model_dir / "task_accuracy.csv"
    deltas_path = model_dir / "task_deltas.csv"
    if not (accuracy_path.is_file() and deltas_path.is_file()):
        if model_id in SUMMARY_SOURCES:
            print(f"Aggregate-only recovery; per-task charts omit {model_id}")
        continue
    accuracy = pd.read_csv(accuracy_path)
    accuracy["model"] = model_id
    accuracy_frames.append(accuracy)
    deltas = pd.read_csv(deltas_path)
    deltas["model"] = model_id
    delta_frames.append(deltas)
    DETAILED_MODELS.append(model_id)
if accuracy_frames:
    accuracy_all = pd.concat(accuracy_frames, ignore_index=True)
    deltas_all = pd.concat(delta_frames, ignore_index=True)
    accuracy_all.to_csv(f"{RESULTS_ROOT}/combined_task_accuracy.csv", index=False)
    deltas_all.to_csv(f"{RESULTS_ROOT}/combined_task_deltas.csv", index=False)

    lambada = accuracy_all[accuracy_all["task"] == "lambada_openai"].set_index("model")[["original", "filtered", "restyled"]]
    axis = lambada.plot(kind="bar", figsize=(10, 5), title="LAMBADA raw accuracy (models with recovered task files)")
    axis.set_xlabel("")
    axis.set_ylabel("Raw accuracy")
    axis.tick_params(axis="x", rotation=20)
    axis.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

    print("LAMBADA comparison")
    display(lambada.style.format("{:.4f}"))
    print("All raw per-task accuracies")
    display(accuracy_all.set_index(["model", "task"]).style.format("{:.4f}"))
    print("All bundle deltas")
    display(deltas_all.set_index(["model", "task"]).style.format("{:+.4f}"))
else:
    print("No detailed task files are available yet; aggregate charts above are still complete.")

if full_val_path.is_file():
    full_val = json.loads(full_val_path.read_text())
    full_val_bpb = full_val["bpb"]["val"]
    display(pd.DataFrame([{"model": CLEAN_MODEL_ID, "validation_tokens": FULL_VAL_TOKENS, "full_val_bpb": full_val_bpb}]))
    axis = pd.Series({CLEAN_MODEL_ID: full_val_bpb}).plot(kind="bar", figsize=(8, 4), title="Full validation BPB")
    axis.set_ylabel("Bits per byte (lower is better)")
    axis.tick_params(axis="x", rotation=15)
    axis.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("No full clean-1930s validation BPB result found yet.")

## 6. Append every clean-1930s CORE result to its existing W&B run

This resumes the training run instead of creating a new project or run. It logs native and Common-20 aggregates, runtime, every raw task accuracy, every centered task score, and one complete task table. If full validation BPB exists, that value is synchronized to the same run summary.

In [ ]:
if LOG_TO_WANDB:
    import re
    import wandb
    assert os.environ.get("WANDB_API_KEY"), "Add WANDB_API_KEY to Colab Secrets."
    clean_core_dir = Path(RESULTS_ROOT) / CLEAN_MODEL_ID
    required_core_files = [clean_core_dir / f"{bundle}.json" for bundle in ("original", "filtered", "restyled")]
    missing = [str(path) for path in required_core_files if not path.is_file()]
    if missing:
        print("Skipping W&B CORE append; clean-model CORE files are missing:", missing)
    else:
        summary_lookup = pd.read_csv(clean_core_dir / "summary.csv").set_index("bundle")
        scalar_metrics = {}
        task_rows = []
        for bundle in ("original", "filtered", "restyled"):
            payload = json.loads((clean_core_dir / f"{bundle}.json").read_text())
            prefix = f"eval/vintage_core/{bundle}"
            scalar_metrics[f"{prefix}/native_core"] = float(payload["core_metric"])
            scalar_metrics[f"{prefix}/common_20_core"] = float(summary_lookup.loc[bundle, "common_20_core"])
            scalar_metrics[f"{prefix}/runtime_seconds"] = float(payload["runtime_seconds"])
            for task, raw_accuracy in payload["results"].items():
                centered_score = float(payload["centered_results"][task])
                task_slug = re.sub(r"[^A-Za-z0-9_.-]+", "_", task).strip("_")
                scalar_metrics[f"{prefix}/raw/{task_slug}"] = float(raw_accuracy)
                scalar_metrics[f"{prefix}/centered/{task_slug}"] = centered_score
                task_rows.append([bundle, task, float(raw_accuracy), centered_score])
        if full_val_path.is_file():
            full_val_payload = json.loads(full_val_path.read_text())
            scalar_metrics["eval/full_val_bpb"] = float(full_val_payload["bpb"]["val"])
            scalar_metrics["full_val_bpb"] = scalar_metrics["eval/full_val_bpb"]
        run = wandb.init(
            entity=CLEAN_WANDB_ENTITY, project=CLEAN_WANDB_PROJECT,
            id=CLEAN_WANDB_RUN_ID, resume="allow", name=CLEAN_MODEL_ID,
        )
        task_table = wandb.Table(
            columns=["bundle", "task", "raw_accuracy", "centered_score"],
            data=task_rows,
        )
        run.log({**scalar_metrics, "eval/vintage_core/task_results": task_table})
        run.summary.update(scalar_metrics)
        run.finish()
        print(f"Appended {len(scalar_metrics)} scalar metrics and {len(task_rows)} task rows to W&B run {CLEAN_WANDB_RUN_ID}.")
else:
    print("W&B logging disabled; local JSON/CSV files remain canonical.")

## 7. Locate or download result files

The ZIP is always created. With the default Drive setting, the uncompressed files also remain available after the Colab runtime ends. Set `DOWNLOAD_RESULTS=True` in the configuration cell if you want the browser download dialog.

In [ ]:
print("Saved results directory:", RESULTS_ROOT)
saved_files = sorted(str(path.relative_to(RESULTS_ROOT)) for path in Path(RESULTS_ROOT).rglob("*") if path.is_file())
print("Saved files:")
for name in saved_files:
    print(" -", name)
archive = shutil.make_archive("/content/vintage-core-results", "zip", RESULTS_ROOT)
print("ZIP archive:", archive)
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive)